# Nested feature-selection revision

This notebook audits the continuation of feature-selection-2.2 without rewriting its frozen negative result. Inner feature ranking uses 2017–2020 only, while 2021–2022 is reserved for outer feature-count and stopping decisions.


## Locate nested artifacts

The canonical runner writes the nested revision under a separate artifact root. Until the full run completes, this notebook reports that state without falling back to the original test-consumed selection.

In [1]:
from pathlib import Path
import json

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "data" / "splits").is_dir():
        PROJECT_ROOT = candidate
        break
EXP_DIR = PROJECT_ROOT / "notebooks/experiment/derived_8.2-feature-selection-2.2"
ARTIFACT_ROOT = EXP_DIR / "artifacts/nested"
print("Nested artifacts available:", ARTIFACT_ROOT.exists())

Nested artifacts available: True


## Compare inner and outer decisions

The outer table is the key safeguard added after the first 2.2 run. Candidate lists are generated entirely inside the train period, then scored on disjoint future years and held-out station groups.

In [2]:
import sys

sys.path.insert(0, str(EXP_DIR))
from generate_results import load_selection_summary

summary = load_selection_summary("nested")
display(pd.DataFrame(summary["datasets"]))
print("Source:", ARTIFACT_ROOT / "selection_summary.json")

,dataset,global_n_features,outer_stopping_reason,regimes
0,derived_8.0,40,minimum_outer_upper_confidence_bound,{}
1,derived_8.2,50,minimum_outer_upper_confidence_bound,"{'0': {'n_features': 65, 'n_delta': 15, 'outer..."


Source: /scratch/user/u.rp352032/MDR-Project/notebooks/experiment/derived_8.2-feature-selection-2.2/artifacts/nested/selection_summary.json


## Locked learner, crossed folds, and MoE ablation

The architectural diagnostics below are retrospective only. They compare the locked final learner, independently generated forward-time and station/time paths, progressive elimination, and the shared-only versus shared-plus-delta MoE. No feature list is seeded or bypassed.

In [3]:
from generate_results import build_candidate_ceiling_table

ceiling = build_candidate_ceiling_table()
display(ceiling)
print("Sources: artifacts/*/candidate_diagnostics/global_candidates.csv")

,artifact_set,dataset,outer_selected_n_features,outer_selected_retrospective_R2,retrospective_ceiling_n_features,retrospective_ceiling_R2
2,crossed_candidates_locked_outer,derived_8.0,100,0.780809,150,0.795770
3,crossed_candidates_locked_outer,derived_8.2,80,0.619727,100,0.675726
0,nested,derived_8.0,100,0.780809,100,0.780809
1,nested,derived_8.2,100,0.675726,100,0.675726
4,progressive_crossed_locked_outer,derived_8.0,100,0.780809,150,0.816924


Sources: artifacts/*/candidate_diagnostics/global_candidates.csv


In [4]:
from generate_results import build_moe_table

moe = build_moe_table()
display(moe)
print("Source:", ARTIFACT_ROOT / "retrospective_test_eval/metrics_summary.csv")

,artifact_set,dataset,model,beta,R2,RMSE,ubRMSE,Bias,MAE,Med|Err|,Pearson
19,nested,derived_8.2,2.2_clustering_dynamic_k2_shared_plus_delta,0.0,0.623748,0.064591,0.063285,-0.012924,0.048296,0.036710,0.808755
20,nested,derived_8.2,2.2_clustering_frozen_k2_shared_only,0.0,0.608959,0.065849,0.064769,-0.011875,0.049063,0.037234,0.799016
21,nested,derived_8.2,2.2_clustering_refit_k2_shared_plus_delta,0.0,0.618213,0.065065,0.063774,-0.012894,0.048600,0.037181,0.806331
16,nested,derived_8.2,2.2_global,0.0,0.662752,0.061152,0.059610,-0.013647,0.046254,0.035500,0.828629


Source: /scratch/user/u.rp352032/MDR-Project/notebooks/experiment/derived_8.2-feature-selection-2.2/artifacts/nested/retrospective_test_eval/metrics_summary.csv
